## 2. Анализ персонала и ФОТ

**Источник данных:** два Excel-файла с графиками работы сотрудников  
**Проблемы при загрузке:** нестандартная структура (каждый сотрудник — 3 строки),  
смешанные форматы, несколько блоков на одном листе, аномальные значения часов  
**Итоговый датасет:** 40 сотрудников, 18 месяцев, ноябрь 2024 — май 2026


In [ ]:
import pandas as pd

# Смотрим листы в обоих файлах
for fname in ['../data/График 2025.xlsx', '../data/График 2026.xlsx']:
    xl = pd.ExcelFile(fname)
    print(f'=== {fname} ===')
    print(f'Листы: {xl.sheet_names}')
    print()

In [ ]:
df_raw = pd.read_excel('../data/График 2025.xlsx',
                       sheet_name='Ноябрь',
                       header=None)

print(f'Размер: {df_raw.shape}')
print()
print(df_raw.head(30).to_string())

In [ ]:
import pandas as pd
import numpy as np
import re

def parse_schedule_sheet(fname, sheet_name, year_month):
    """
    Парсит один лист графика.
    year_month — строка типа '2024-11' для привязки месяца.
    """
    df_raw = pd.read_excel(fname, sheet_name=sheet_name, header=None)
    
    records = []
    i = 0
    
    while i < len(df_raw):
        row = df_raw.iloc[i]
        
        # Ищем строку с именем сотрудника
        # Признак: в колонке 0 есть текст, это не 'Имя/Дата', не NaN
        name_val = str(row[0]).strip().replace('\n', ' ').strip()
        
        if (pd.notna(row[0]) and 
            name_val not in ['', 'nan', 'Имя/Дата', 'имя/дата', 
                              'Имя\n/\nДата', 'имя\n/\nдата'] and
            not name_val.lower().startswith('имя')):
            
            # Это строка с именем — берём итого часов и зарплату
            # Итого часов — колонка 30 (предпоследняя)
            # Зарплата — колонка 32 (последняя)
            
            # Ищем строку с часами (через 2 строки)
            if i + 2 < len(df_raw):
                hours_row = df_raw.iloc[i + 2]
                total_hours = hours_row[30] if pd.notna(hours_row[30]) else None
                salary = row[32] if pd.notna(row[32]) else None
                
                # Чистим имя
                clean = name_val.replace('\n', ' ').strip()
                # Убираем множественные пробелы
                clean = re.sub(r'\s+', ' ', clean)
                
                if clean and total_hours is not None:
                    records.append({
                        'name':        clean,
                        'year_month':  year_month,
                        'total_hours': pd.to_numeric(total_hours, errors='coerce'),
                        'salary':      pd.to_numeric(
                                           str(salary).replace(' ', ''), 
                                           errors='coerce') if salary else None,
                    })
        i += 1
    
    return pd.DataFrame(records)

# Тест на одном листе
df_test = parse_schedule_sheet('../data/График 2025.xlsx', 'Ноябрь', '2024-11')
print(f'Сотрудников найдено: {len(df_test)}')
print()
print(df_test.to_string())

In [ ]:
# Смотрим строки вокруг одного сотрудника — сотрудника
df_raw = pd.read_excel('../data/График 2025.xlsx',
                       sheet_name='Ноябрь', header=None)

# Печатаем строки 2-7 — там должна быть сотрудника
for idx in range(2, 8):
    row = df_raw.iloc[idx]
    print(f'Строка {idx}: col[0]={repr(str(row[0]))[:20]}  '
          f'col[30]={repr(row[30])}  col[31]={repr(row[31])}  '
          f'col[32]={repr(row[32])}')

In [ ]:
def parse_schedule_sheet(fname, sheet_name, year_month):
    df_raw = pd.read_excel(fname, sheet_name=sheet_name, header=None)
    records = []
    i = 0

    skip_names = ['', 'nan', 'часов', '\nчасов\n']

    while i < len(df_raw):
        row = df_raw.iloc[i]
        name_val = re.sub(r'\s+', ' ', str(row[0])).strip()

        is_header = name_val.lower().startswith('имя')
        is_skip   = name_val in skip_names

        if pd.notna(row[0]) and not is_header and not is_skip:

            salary = pd.to_numeric(
                str(row[32]).replace(' ', ''), errors='coerce')

            # Итого часов — col[31] через 2 строки
            total_hours = None
            if i + 2 < len(df_raw):
                hours_row   = df_raw.iloc[i + 2]
                total_hours = pd.to_numeric(hours_row[31], errors='coerce')

            if name_val and pd.notna(salary):
                records.append({
                    'name':        name_val,
                    'year_month':  year_month,
                    'total_hours': total_hours,
                    'salary':      salary,
                })
        i += 1

    return pd.DataFrame(records)

# Тест
df_test = parse_schedule_sheet(
    '../data/График 2025.xlsx', 'Ноябрь', '2024-11')
print(f'Сотрудников: {len(df_test)}')
print()
print(df_test.to_string())

In [ ]:
sheets_2025 = {
    'Ноябрь':       '2024-11',
    'Декабрь':      '2024-12',
    'Январь':       '2025-01',
    'Февраль':      '2025-02',
    'Март ':        '2025-03',
    'Апрель':       '2025-04',
    'Май':          '2025-05',
    'Июнь':         '2025-06',
    'Июль':         '2025-07',
    'Август':       '2025-08',
    'Сентябрь':     '2025-09',
    'Октябрь':      '2025-10',
    'Ноябрь ':      '2025-11',
    'Декабрь ':     '2025-12',
    'Январь 2026':  '2026-01',
}

sheets_2026 = {
    'Январь ':  '2026-01',
    'Февраль':  '2026-02',
    'Март':     '2026-03',
    'Апрель':   '2026-04',
    'Май':      '2026-05',
}

all_dfs = []

for sheet, ym in sheets_2025.items():
    try:
        df_s = parse_schedule_sheet('../data/График 2025.xlsx', sheet, ym)
        if len(df_s) > 0:
            all_dfs.append(df_s)
            print(f'✓ {sheet:20} → {ym} | {len(df_s)} сотрудников')
    except Exception as e:
        print(f'✗ {sheet:20} → ошибка: {e}')

for sheet, ym in sheets_2026.items():
    try:
        df_s = parse_schedule_sheet('../data/График 2026.xlsx', sheet, ym)
        if len(df_s) > 0:
            all_dfs.append(df_s)
            print(f'✓ {sheet:20} → {ym} | {len(df_s)} сотрудников')
    except Exception as e:
        print(f'✗ {sheet:20} → ошибка: {e}')

df_staff = pd.concat(all_dfs, ignore_index=True)
df_staff['year_month'] = pd.to_datetime(df_staff['year_month'])

print(f'\nИтого записей: {len(df_staff)}')
print(f'Период: {df_staff["year_month"].min().strftime("%Y-%m")} — '
      f'{df_staff["year_month"].max().strftime("%Y-%m")}')
print(f'Уникальных сотрудников: {df_staff["name"].nunique()}')

In [ ]:
# Убираем дубли по имя + месяц — оставляем первую запись
df_staff = df_staff.drop_duplicates(subset=['name', 'year_month'])

# Норма часов в месяц
NORM_HOURS = 168

# Добавляем метрики
df_staff['overtime'] = df_staff['total_hours'] - NORM_HOURS
df_staff['is_overtime'] = df_staff['overtime'] > 0
df_staff['hour_rate'] = (df_staff['salary'] / df_staff['total_hours'].replace(0, np.nan))

print(f'Записей после дедупликации: {len(df_staff)}')
print()
print('Общая статистика по часам:')
print(df_staff['total_hours'].describe().round(1))
print()
print('Общая статистика по зарплате:')
print(df_staff['salary'].describe().round(0))

In [ ]:
# Смотрим записи с подозрительными часами
print('Записи с часами > 400 (явный мусор):')
print(df_staff[df_staff['total_hours'] > 400][
    ['name','year_month','total_hours','salary']
].to_string())

print()
print('Записи с часами = 0 или NaN:')
print(df_staff[df_staff['total_hours'].isna() | 
               (df_staff['total_hours'] == 0)][
    ['name','year_month','total_hours','salary']
].to_string())

In [ ]:
# Убираем служебные строки
junk_names = ['часов', 'часы', 'ЧАСЫ', 'ЧАСОВ', 'Часов',
              'час', 'ЧАС']

df_clean = df_staff[
    ~df_staff['name'].str.lower().str.strip().isin(
        [j.lower() for j in junk_names])
].copy()

# Убираем аномальные часы — реальный диапазон 0–400
df_clean = df_clean[
    df_clean['total_hours'].isna() | 
    (df_clean['total_hours'] <= 400)
].copy()

# Убираем нулевые зарплаты (неактивные записи)
df_clean = df_clean[df_clean['salary'] > 0].copy()

import hashlib

def anonymize_name(name):
    h = hashlib.md5(name.encode()).hexdigest()[:6].upper()
    return f'Сотрудник_{h}'

df_clean['name'] = df_clean['name'].apply(anonymize_name)

print(f'Записей после очистки: {len(df_clean)}')

print(f'Записей после очистки: {len(df_clean)}')
print()
print('Статистика по часам (где есть):')
hours_valid = df_clean['total_hours'].dropna()
print(f'  Записей с часами: {len(hours_valid)}')
print(f'  Среднее: {hours_valid.mean():.0f} ч')
print(f'  Медиана: {hours_valid.median():.0f} ч')
print(f'  Мин: {hours_valid.min():.0f}  Макс: {hours_valid.max():.0f}')
print()
print('Статистика по зарплате:')
print(f'  Средняя: {df_clean["salary"].mean():,.0f} ₽')
print(f'  Медиана: {df_clean["salary"].median():,.0f} ₽')
print(f'  Мин: {df_clean["salary"].min():,.0f}  '
      f'Макс: {df_clean["salary"].max():,.0f}')
print()
print('Уникальных сотрудников:', df_clean['name'].nunique())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Анализ персонала — Ресторан Искусство, ноя 2024 – май 2026',
             fontsize=14, fontweight='bold', y=1.01)

# ── График 1: ФОТ по месяцам ─────────────────────────────
fot_by_month = (df_clean.groupby('year_month')['salary']
                .sum().reset_index()
                .sort_values('year_month'))

axes[0].bar(range(len(fot_by_month)),
            fot_by_month['salary'] / 1000,
            color='steelblue', edgecolor='white')
axes[0].set_xticks(range(len(fot_by_month)))
axes[0].set_xticklabels(
    [d.strftime('%b\n%y') for d in fot_by_month['year_month']],
    fontsize=8)
for i, val in enumerate(fot_by_month['salary'].values):
    axes[0].text(i, val/1000 + 5, f'{val/1000:.0f}к',
                 ha='center', fontsize=7)
axes[0].set_title('ФОТ по месяцам')
axes[0].set_ylabel('Тыс. руб')
axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

# ── График 2: часы по сотрудникам ────────────────────────
hours_data = (df_clean[df_clean['total_hours'].notna() &
                        (df_clean['total_hours'] > 0)]
              .groupby('name')['total_hours']
              .mean()
              .sort_values(ascending=True))

colors_h = ['salmon' if h > 168 else 'steelblue'
            for h in hours_data.values]

axes[1].barh(hours_data.index, hours_data.values,
             color=colors_h, edgecolor='white')
axes[1].axvline(168, color='red', linestyle='--',
                linewidth=1.5, label='Норма 168ч')
axes[1].set_title('Средние часы по сотруднику\n(красный = переработка)')
axes[1].set_xlabel('Часов в месяц (среднее)')
axes[1].legend(fontsize=9)
axes[1].grid(axis='x', alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)
axes[1].tick_params(axis='y', labelsize=8)

# ── График 3: топ по зарплате ─────────────────────────────
salary_top = (df_clean.groupby('name')['salary']
              .mean()
              .sort_values(ascending=True)
              .tail(15))

bars3 = axes[2].barh(salary_top.index,
                     salary_top.values / 1000,
                     color='steelblue', edgecolor='white')
for bar, val in zip(bars3, salary_top.values):
    axes[2].text(bar.get_width() + 0.5,
                 bar.get_y() + bar.get_height()/2,
                 f'{val/1000:.0f}к', va='center', fontsize=9)
axes[2].set_title('Топ-15 по средней зарплате')
axes[2].set_xlabel('Тыс. руб (среднее за период)')
axes[2].set_xlim(0, salary_top.max()/1000 * 1.25)
axes[2].grid(axis='x', alpha=0.3)
axes[2].spines[['top','right']].set_visible(False)
axes[2].tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.show()

## Выводы — Анализ персонала

**Данные:** 18 месяцев, 40 сотрудников, ноябрь 2024 – май 2026

### ФОТ
- Пик: декабрь 2024 (860к ₽) — без учета управляющего состава
- После января 2025 ФОТ снизился в 2 раза — сокращение штата
- Стабилизация на уровне 400 тыс с середины 2025

### Часы и переработки
- есть ряд сотрудников которые перерабатывают системную рабочую норму часов
- Большинство сотрудников укладываются в норму 168 ч
- Переработки = скрытые затраты, не отражённые в окладе

### Зарплаты
- Разброс: от 40 тыс. до 120 тыс. руб.
- Средняя по команде: 80 тыс. руб.